In [4]:
from tabulate import tabulate

In [19]:
headers = ['Tokenizer', 'Vocab Size', 'Test Set Language', 'Fertility (tokens/word)', 'Chars/Token', 'Fragment Rate', 
           'Renyi Entropy (alpha=2.5) bits', 'Renyi Efficiency']

In [32]:
# 
data = [['BPE', '4000', 'Kannada', '2.32596', '3.47562', '0.00519', '9.1843', '0.755']]
data.append(['BPE', '4000', 'English', '1.28049', '4.79857', '0.00136', '', ''])

data.append(['BPE_(constrained)', '4000\n(3998)', 'Kannada', '2.31233', '3.4961', '0.00295', '9.2934', '0.7551'])
data.append(['BPE_(constrained)', '4000\n(3998)', 'English', '1.28049', '4.79857', '0.00136', '', ''])
# These metrics are actually for a BPE tokenizer that used 3998 merges, since that's the number of tokens in the Unigram tokenizer

data.append(['Unigram', '4000\n(3508)', 'Kannada', '2.60124139', '3.107797', '0.13749', '6.8529', '0.5753'])
data.append(['Unigram', '4000\n(3508)', 'English', '1.28049', '4.79857', '0.00136', '', ''])
# Of the 4,000 tokens, the number of Kannada tokens is only 3,508

data.append(['Unigram_(char)', '4000\n(3998)', 'Kannada', '2.313', '3.49515', '0.00876', '7.7949', '0.6374'])
data.append(['Unigram_(char)', '4000\n(3998)', 'English', '1.28049', '4.79857', '0.00136', '', ''])
# The other two were already in OLMo's tokenizer

data.append(['Unigram', '6000\n(5357)', 'Kannada', '2.4517150', '3.2973377', '0.16339', '', ''])
data.append(['Unigram', '6000\n(5357)', 'English', '1.280489', '4.79857', '0.00136', '', ''])

data.append(['BPE', '6000', 'Kannada', '2.14098', '3.77591', '0.00456', '', ''])
data.append(['BPE', '6000', 'English', '1.28049', '4.79857', '0.00136', '', ''])


In [33]:
print(tabulate(data, headers=headers, tablefmt="grid"))

+-------------------+--------------+---------------------+---------------------------+---------------+-----------------+----------------------------------+--------------------+
| Tokenizer         | Vocab Size   | Test Set Language   |   Fertility (tokens/word) |   Chars/Token |   Fragment Rate |   Renyi Entropy (alpha=2.5) bits |   Renyi Efficiency |
+===================+==============+=====================+===========================+===============+=================+==================================+====================+
| BPE               | 4000         | Kannada             |                   2.32596 |       3.47562 |         0.00519 |                           9.1843 |             0.755  |
+-------------------+--------------+---------------------+---------------------------+---------------+-----------------+----------------------------------+--------------------+
| BPE               | 4000         | English             |                   1.28049 |       4.79857 |         0.00

Note: 
- The training set for the code-point-level Unigram tokenizer had 300,000 data points. The character-level Unigram tokenizer had the full ~36 million data points.
<br><br>
- The Renyi Entropy and Efficiency were calculated on a 22k document sample, whereas the fertility and fragment rate were calculated on a 2k evaluation set.
<br><br>
- The fertility reported in the above table is calculated as total number of tokens divided by whitespace-separated words. The words are from the Kannada eval set, including punctuation and a handful of non-Kannada text. In the notebook "5_comparing_new_bpe_unigram...", the fertility is calculated by restricting the words considered to only Kannada pre tokens.

#### Conclusions

1. The BPE tokenizer with 4,000 new merges has a fertility of 2.3260 tokens per word on the Kannada evaluation set. On the English evaluation set, its fertility is 1.2805, which is identical to the base tokenizer (0 new merges). This holds for every tokenizer that is being compared in the above table: English fertility, characters per token, and fragment rate are unchanged, so adding Kannada tokens did not alter how English text is tokenized.
<br><br>

1. The BPE tokenizer trained with the syllable constraint has a fertility of 2.3123 with its first 3,9998 merges (to match the Unigram vocabulary). The constraint was that no token may start with a split vowel sign, virama or anusvara. Its fertility is slightly lower than the unconstrained tokenizer's 2.3260, despite having two fewer tokens. The constraint did not cost any compression and resulted in a small improvement in fertility and fragment rate.
<br><br>

1. The byte-level Unigram tokenizer, trained with a vocabulary size of 4,000 (of which 3,508 are new Kannada tokens), has a fertility of 2.6012. It has about 500 fewer new Kannada tokens than the BPE tokenizers. Since it was trained on byte-level data, it often splits characters across tokens (a fragment rate of 13–16%), which increases the fertility.
<br><br>

1. The Unigram tokenizer trained on Unicode characters instead of bytes (3,998 new Kannada tokens) has a fertility of 2.313, effectively identical to the constrained BPE tokenizer at the same size (2.3123), and 11% lower than the byte-level Unigram tokenizer. Two changes contributed to this: training on characters, which eliminated the byte fragments, and a larger share of the vocabulary going to Kannada tokens (3,998 vs 3,508). With matched vocabulary sizes, the BPE and Unigram methods now reach the same compression.